# MallWorld Data Exploration: Looking for Non-Obvious Patterns

**Goal**: Explore the data to find patterns that could discriminate between competing hypotheses—WITHOUT pre-deciding what to look for.

The key question: What's in this data that might surprise us? What relationships exist that common sense would NOT predict?

Great question. Let me explain what these massive χ² values actually mean and what that implies for MallWorld research.

## What the Massive Chi-Square Values Mean

### The Basic Logic

Chi-square tests whether two variables are **independent** (unrelated) or **associated** (systematically co-occur). The null hypothesis is: "These things are randomly paired—no relationship."

| Finding | χ² | p-value | What It Means |
|---------|-----|---------|---------------|
| Location-atmosphere | 1,677.05 | < 10⁻¹²⁶ | The odds this pattern is random chance: **1 in 10¹²⁶** |
| Entity-atmosphere | 792.02 | < 10⁻¹⁰⁸ | The odds this pattern is random chance: **1 in 10¹⁰⁸** |
| Vertical-atmosphere | 143.1 | < 10⁻²¹ | The odds this pattern is random chance: **1 in 10²¹** |

For comparison:
- There are approximately 10⁸⁰ atoms in the observable universe
- 10⁻¹²⁶ is **46 orders of magnitude smaller** than picking one specific atom at random

### What This Actually Tells Us

**The patterns are real, not noise.** When you see χ² = 1,677 with p < 10⁻¹²⁶, you're looking at structure that cannot plausibly arise from random dream generation. Something is organizing this data.

But here's the key insight: **chi-square tells you the pattern exists, not what causes it.**

The massive values establish:
1. ✅ MallWorld dreams have **non-random internal structure**
2. ✅ Location types systematically predict atmospheres
3. ✅ Entity types systematically predict emotional environments
4. ✅ Vertical position systematically predicts atmosphere

They do NOT establish:
- ❓ WHY these patterns exist
- ❓ Whether the patterns are cultural (learned) or ontological (inherent)
- ❓ Whether dreamers are perceiving something real or constructing something consistent

### The Research Opportunity

This is where MallWorld gets interesting. The chi-square values have done their job—they've proven structure exists. Now the question becomes: **What kind of structure is it?**

There are three competing explanations:

| Explanation | Prediction | How to Test |
|-------------|------------|-------------|
| **Cultural Construction** | Patterns should vary by demographic (age, culture, religion) | Cross-cultural analysis; demographic stratification |
| **Psychological Projection** | Patterns should correlate with individual traits (anxiety, trauma) | Within-dreamer consistency; psychological profiling |
| **Correspondential Ontology** | Patterns should be universal regardless of demographics; same structure as NDE/myth | Cross-cultural consistency; comparison to independent consciousness phenomena |

### Where to Go Deeper

Given that structure exists, the next analytical steps would be:

1. **Cross-Cultural Consistency Test**: Do non-Western MallWorld dreamers show the same location-atmosphere mappings? If correspondence is ontological, it shouldn't vary by culture.

2. **Within-Dreamer Longitudinal Analysis**: Do individual dreamers maintain consistent correspondential patterns across multiple dreams? This tests whether the structure is perceived (consistent) vs constructed (variable).

3. **Comparison to Independent Phenomena**: How closely do MallWorld entity patterns match NDE entity patterns? Identical function differentiation across dream and near-death states would be strong evidence for perception of the same reality.

4. **Prediction Testing**: The framework predicts specific correspondences (fire = love/passion, water = truth, etc.). Can we find explicit symbolic content in dreams that tests these predictions?

5. **Effect Size Analysis**: Chi-square tells us patterns exist but not how STRONG they are. Cramér's V or similar measures would tell us what percentage of atmosphere variation is explained by location/entity type.

Would you like me to explore any of these directions, or look at what data you already have that could support these deeper analyses?

In [4]:
import json
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu
from collections import Counter, defaultdict
from pathlib import Path
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD RAW STRUCTURED DATA (Full control over all fields)
# ============================================================================

def load_mallworld_data(structured_path: Path) -> List[Dict]:
    """Load all extracted Mall World JSON files."""
    records = []
    for json_file in sorted(structured_path.glob('*.json')):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                record = json.load(f)
                records.append(record)
        except Exception as e:
            print(f"Error loading {json_file.name}: {e}")
    return records

# Load data
structured_path = Path('../structured')
raw_records = load_mallworld_data(structured_path)
print(f"✓ Loaded {len(raw_records):,} total records")

# Filter to actual dream reports
VALID_POST_TYPES = {'dream_report', 'dream_report_with_map'}

def is_valid_dream_report(record: Dict) -> bool:
    ext = record.get('extraction', {}) or {}
    classification = ext.get('classification', {}) or {}
    post_type = classification.get('post_type', '')
    if post_type not in VALID_POST_TYPES:
        return False
    if classification.get('should_skip_extraction', False):
        return False
    locations = ext.get('locations', [])
    if not locations or len(locations) == 0:
        return False
    return True

valid_records = [r for r in raw_records if is_valid_dream_report(r)]
print(f"✓ After filtering: {len(valid_records):,} valid dream reports")

✓ Loaded 3,732 total records
✓ After filtering: 1,926 valid dream reports


In [5]:
# ============================================================================
# FLATTEN DATA INTO DATAFRAMES (with full field access)
# ============================================================================

def get_dream_id(record):
    """Extract dream ID from record."""
    return record.get('source_url', '').split('/')[-2] if record.get('source_url') else record.get('source_file', '')

# Build locations DataFrame with ALL fields
locations_data = []
for record in valid_records:
    dream_id = get_dream_id(record)
    ext = record.get('extraction', {}) or {}
    meta = ext.get('meta', {}) or {}
    author = meta.get('author', {}) or {}
    
    for loc in ext.get('locations', []):
        qualities = loc.get('qualities', {}) or {}
        position = loc.get('position', {}) or {}
        
        locations_data.append({
            'dream_id': dream_id,
            'location_id': loc.get('location_id'),
            'location_type': loc.get('location_type', 'unspecified'),
            'location_name': loc.get('location_name'),
            # Position
            'vertical_position': position.get('vertical', 'not_mentioned'),
            'horizontal_position': position.get('horizontal', 'not_mentioned'),
            'cardinal_direction': position.get('cardinal', 'not_mentioned'),
            # Core qualities
            'light': qualities.get('light', 'not_mentioned'),
            'light_temperature': qualities.get('light_temperature', 'not_mentioned'),
            'time_of_day': qualities.get('time_of_day', 'not_mentioned'),
            'state': qualities.get('state', 'not_mentioned'),
            'atmosphere': qualities.get('atmosphere', 'not_mentioned'),
            'reality_stability': qualities.get('reality_stability', 'not_mentioned'),
            'crowding': qualities.get('crowding', 'not_mentioned'),
            'crowd_behavior': qualities.get('crowd_behavior', 'not_mentioned'),
            'cleanliness': qualities.get('cleanliness', 'not_mentioned'),
            'privacy_status': qualities.get('privacy_status', 'not_mentioned'),
            # Dreamer response
            'affective_response': loc.get('affective_response', 'not_mentioned'),
            'somatic_response': loc.get('somatic_response', 'none'),
            'dreamer_role': loc.get('dreamer_role', 'not_mentioned'),
            'visit_order': loc.get('visit_order'),
            # Demographics from author (THIS IS THE KEY PART)
            'author_gender': author.get('gender', 'not_mentioned'),
            'author_age_range': author.get('age_range', 'not_mentioned'),
            # Dream-level meta
            'recurrence': meta.get('recurrence', 'not_mentioned'),
            'dreamer_identity': meta.get('dreamer_identity', 'not_mentioned'),
        })

df_locations = pd.DataFrame(locations_data)

# Build entities DataFrame
entities_data = []
for record in valid_records:
    dream_id = get_dream_id(record)
    ext = record.get('extraction', {}) or {}
    meta = ext.get('meta', {}) or {}
    author = meta.get('author', {}) or {}
    
    for interaction in ext.get('interactions', []):
        loc_id = interaction.get('location_id')
        for entity in interaction.get('entities_involved', []):
            entities_data.append({
                'dream_id': dream_id,
                'location_id': loc_id,
                'entity_type': entity.get('entity_type', 'other'),
                'demeanor': entity.get('demeanor', 'not_mentioned'),
                'role': entity.get('role', 'not_mentioned'),
                'authority_nature': entity.get('authority_nature', 'not_mentioned'),
                'is_known': entity.get('is_known', 'not_mentioned'),
                'is_recurring': entity.get('is_recurring', 'not_mentioned'),
                # Demographics
                'author_gender': author.get('gender', 'not_mentioned'),
                'author_age_range': author.get('age_range', 'not_mentioned'),
            })

df_entities = pd.DataFrame(entities_data)

# Build connections DataFrame
connections_data = []
for record in valid_records:
    dream_id = get_dream_id(record)
    ext = record.get('extraction', {}) or {}
    
    for conn in ext.get('connections', []):
        connections_data.append({
            'dream_id': dream_id,
            'from_location_id': conn.get('from_location_id'),
            'to_location_id': conn.get('to_location_id'),
            'connection_type': conn.get('connection_type', 'unspecified'),
            'transit_mode': conn.get('transit_mode', 'not_mentioned'),
            'direction': conn.get('direction', 'unknown'),
            'difficulty': conn.get('difficulty', 'not_mentioned'),
            'outcome': conn.get('outcome', 'not_mentioned'),
        })

df_connections = pd.DataFrame(connections_data)

# Build interactions DataFrame
interactions_data = []
for record in valid_records:
    dream_id = get_dream_id(record)
    ext = record.get('extraction', {}) or {}
    
    for interaction in ext.get('interactions', []):
        interactions_data.append({
            'dream_id': dream_id,
            'location_id': interaction.get('location_id'),
            'interaction_type': interaction.get('interaction_type', 'other'),
            'outcome': interaction.get('outcome', 'not_mentioned'),
            'failure_type': interaction.get('failure_type', 'not_applicable'),
            'n_entities': len(interaction.get('entities_involved', [])),
        })

df_interactions = pd.DataFrame(interactions_data)

print(f"\n✓ DataFrames built:")
print(f"  Locations:    {len(df_locations):,}")
print(f"  Entities:     {len(df_entities):,}")
print(f"  Connections:  {len(df_connections):,}")
print(f"  Interactions: {len(df_interactions):,}")


✓ DataFrames built:
  Locations:    8,707
  Entities:     3,704
  Connections:  4,600
  Interactions: 6,075


## Rigorous Tests for “Ontological vs Cultural”

This section implements **preregisterable, falsifiable** statistical tests aligned with the deeper-analysis directions in Cell 2.

**Key design choices**
- Use *effect sizes* (Cramér’s $V$, rank correlations) plus uncertainty (bootstrap CIs), not just p-values.
- Use *holdout / stability checks* (time splits and random splits) so we don’t overfit a single corpus.
- Use *negative controls* (label permutation) to guard against spurious structure.
- Use *within-dreamer* analyses using Reddit author metadata (from the raw scraped cases) to separate “idiosyncratic construction” vs “shared stable topology.”

**Important caveat**: demographic fields (`gender`, `age_range`) are likely sparse; this notebook therefore uses additional, available stratifiers (time, content flags, map presence) as “cultural proxy” splits when needed.

In [33]:
# ============================================================================
# ENRICH DREAM-LEVEL METADATA (author, timestamp, engagement)
# ============================================================================

from datetime import datetime
from pathlib import Path
import math
import os

def load_source_case_from_structured_record(record: Dict[str, Any]) -> Dict[str, Any] | None:
    """Load the original scraped case JSON referenced by a structured output record."""
    source_file = record.get('source_file')
    if not source_file:
        return None
    try:
        with open(source_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return None

def parse_iso_datetime(dt_str: str | None) -> datetime | None:
    if not dt_str:
        return None
    try:
        # Handles '...+00:00' format from scraper
        return datetime.fromisoformat(dt_str)
    except Exception:
        return None

# Build a dream-level table: one row per source post (dream)
dream_rows = []
for record in valid_records:
    ext = record.get('extraction', {}) or {}
    cls = ext.get('classification', {}) or {}
    meta = ext.get('meta', {}) or {}

    source_case = load_source_case_from_structured_record(record) or {}
    source_meta = (source_case.get('metadata', {}) or {})

    dream_rows.append({
        'dream_id': get_dream_id(record),
        'source_url': record.get('source_url'),
        'source_id': source_case.get('source_id'),
        'title': source_case.get('title') or record.get('title'),
        'author': source_meta.get('author', None),
        'date_published': parse_iso_datetime(source_case.get('date_published')),
        'post_score': source_meta.get('score', None),
        'num_comments': source_meta.get('num_comments', None),
        'upvote_ratio': source_meta.get('upvote_ratio', None),
        'has_image': bool(source_meta.get('image_urls')),
        'post_hint': source_meta.get('post_hint', None),
        # Classification fields
        'post_type': cls.get('post_type', None),
        'detail_level': cls.get('detail_level', None),
        'content_flags': tuple(cls.get('content_flags', []) or []),
        'has_map_image': cls.get('has_map_image', None),
        'has_extractable_dream': cls.get('has_extractable_dream', None),
        # Dream meta
        'map_coherence': meta.get('map_coherence', None),
        'recurrence': meta.get('recurrence', None),
        'time_flow': meta.get('time_flow', None),
        'dreamer_identity': meta.get('dreamer_identity', None),
        # Author demographics (usually sparse)
        'author_gender': (meta.get('author', {}) or {}).get('gender', 'not_mentioned'),
        'author_age_range': (meta.get('author', {}) or {}).get('age_range', 'not_mentioned'),
        # Size proxies
        'n_locations': len(ext.get('locations', []) or []),
        'n_connections': len(ext.get('connections', []) or []),
        'n_interactions': len(ext.get('interactions', []) or []),
    })

df_dreams = pd.DataFrame(dream_rows)

# Convenience time split columns (for “cultural drift” / stability tests)
df_dreams['year'] = df_dreams['date_published'].dt.year
df_dreams['is_recent'] = df_dreams['year'].ge(df_dreams['year'].median())

print('✓ Dream-level table built')
print(f"  Dreams: {len(df_dreams):,}")
print(f"  Unique authors (non-null): {df_dreams['author'].dropna().nunique():,}")
print(f"  Dreams with known author: {df_dreams['author'].notna().mean()*100:.1f}%")
print(f"  Dreams with known gender: {(df_dreams['author_gender']!='not_mentioned').mean()*100:.1f}%")
print(f"  Dreams with known age range: {(df_dreams['author_age_range']!='not_mentioned').mean()*100:.1f}%")

✓ Dream-level table built
  Dreams: 1,926
  Unique authors (non-null): 1,292
  Dreams with known author: 100.0%
  Dreams with known gender: 1.6%
  Dreams with known age range: 8.6%


In [34]:
# ============================================================================
# STATS UTILITIES: effect sizes, bootstrap CIs, permutation controls
# ============================================================================

def _drop_missing(series: pd.Series, missing=('not_mentioned', None, np.nan)) -> pd.Series:
    return series[~series.isin([m for m in missing if m is not np.nan])].dropna()

def cramers_v_bias_corrected(ct: pd.DataFrame) -> float:
    """Bias-corrected Cramér's V for a contingency table."""
    if ct.size == 0:
        return np.nan
    obs = ct.to_numpy()
    n = obs.sum()
    if n == 0:
        return np.nan
    chi2, _, _, _ = chi2_contingency(obs, correction=False)
    phi2 = chi2 / n
    r, k = obs.shape
    # Bias correction (Bergsma, 2013 style)
    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    rcorr = r - ((r - 1) ** 2) / max(n - 1, 1)
    kcorr = k - ((k - 1) ** 2) / max(n - 1, 1)
    denom = max(min(kcorr - 1, rcorr - 1), 1e-12)
    return float(np.sqrt(phi2corr / denom))

def permutation_chi2_pvalue(ct: pd.DataFrame, n_perm: int = 2000, seed: int = 42) -> float:
    """Permutation p-value for independence using the same margins (label-shuffle).

    This treats the table as counts of paired labels by expanding to samples,
    permuting one label column, then recomputing chi-square. For large tables,
    this can be slow; use moderate n_perm.
    """
    rng = np.random.default_rng(seed)
    obs = ct.to_numpy()
    n = int(obs.sum())
    if n == 0:
        return np.nan
    # Expand to paired samples
    rows = []
    cols = []
    row_labels = list(ct.index)
    col_labels = list(ct.columns)
    for i, rlab in enumerate(row_labels):
        for j, clab in enumerate(col_labels):
            c = int(obs[i, j])
            if c <= 0:
                continue
            rows.extend([rlab] * c)
            cols.extend([clab] * c)
    rows = np.array(rows, dtype=object)
    cols = np.array(cols, dtype=object)
    if len(rows) == 0:
        return np.nan

    def chi2_stat(a, b):
        ctab = pd.crosstab(a, b)
        chi2, _, _, _ = chi2_contingency(ctab, correction=False)
        return chi2

    observed = chi2_stat(rows, cols)
    more_extreme = 0
    cols_perm = cols.copy()
    for _ in range(n_perm):
        rng.shuffle(cols_perm)
        if chi2_stat(rows, cols_perm) >= observed:
            more_extreme += 1
    return float((more_extreme + 1) / (n_perm + 1))

def bootstrap_ci_cramers_v(ct: pd.DataFrame, n_boot: int = 500, seed: int = 123) -> tuple[float, float]:
    """Bootstrap CI for bias-corrected Cramér's V by resampling paired observations."""
    rng = np.random.default_rng(seed)
    obs = ct.to_numpy()
    n = int(obs.sum())
    if n <= 1:
        return (np.nan, np.nan)
    row_labels = np.array(ct.index, dtype=object)
    col_labels = np.array(ct.columns, dtype=object)

    # Expand to paired samples
    pairs = []
    for i, rlab in enumerate(row_labels):
        for j, clab in enumerate(col_labels):
            c = int(obs[i, j])
            if c <= 0:
                continue
            pairs.extend([(rlab, clab)] * c)
    if not pairs:
        return (np.nan, np.nan)
    pairs = np.array(pairs, dtype=object)

    vs = []
    for _ in range(n_boot):
        sample_idx = rng.integers(0, len(pairs), size=len(pairs))
        sample = pairs[sample_idx]
        ct_b = pd.crosstab(sample[:, 0], sample[:, 1])
        vs.append(cramers_v_bias_corrected(ct_b))

    lo, hi = np.nanpercentile(vs, [2.5, 97.5])
    return float(lo), float(hi)

def association_report(
    df: pd.DataFrame,
    x: str,
    y: str,
    *,
    min_total: int = 50,
    drop_not_mentioned: bool = True,
    max_levels: int | None = None,
    n_perm: int = 0,
    seed: int = 42,
    n_boot: int = 0,
    title: str | None = None,
) -> dict:
    """Compute chi-square, bias-corrected Cramér's V, and optional perm/boot measures."""
    dfx = df[[x, y]].copy()
    if drop_not_mentioned:
        dfx = dfx[(dfx[x] != 'not_mentioned') & (dfx[y] != 'not_mentioned')]
    dfx = dfx.dropna()

    # Optionally collapse to top-k levels to avoid huge sparse tables
    if max_levels is not None:
        top_x = dfx[x].value_counts().head(max_levels).index
        top_y = dfx[y].value_counts().head(max_levels).index
        dfx = dfx[dfx[x].isin(top_x) & dfx[y].isin(top_y)]

    ct = pd.crosstab(dfx[x], dfx[y])
    n = int(ct.to_numpy().sum())
    result = {
        'test': title or f"{x} × {y}",
        'x': x,
        'y': y,
        'n': n,
        'levels_x': ct.shape[0],
        'levels_y': ct.shape[1],
        'chi2': np.nan,
        'dof': np.nan,
        'p': np.nan,
        'cramers_v': np.nan,
        'p_perm': np.nan,
        'v_ci_low': np.nan,
        'v_ci_high': np.nan,
    }
    if n < min_total or ct.shape[0] < 2 or ct.shape[1] < 2:
        return result

    chi2, p, dof, expected = chi2_contingency(ct, correction=False)
    result['chi2'] = float(chi2)
    result['p'] = float(p)
    result['dof'] = int(dof)
    result['cramers_v'] = cramers_v_bias_corrected(ct)

    if n_perm and n_perm > 0:
        result['p_perm'] = permutation_chi2_pvalue(ct, n_perm=n_perm, seed=seed)
    if n_boot and n_boot > 0:
        lo, hi = bootstrap_ci_cramers_v(ct, n_boot=n_boot, seed=seed + 1)
        result['v_ci_low'] = lo
        result['v_ci_high'] = hi
    return result

def fdr_bh(p_values: list[float]) -> list[float]:
    """Benjamini–Hochberg FDR correction. Returns q-values in original order."""
    p = np.array([1.0 if (pv is None or np.isnan(pv)) else pv for pv in p_values], dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    q = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = min(prev, ranked[i] * n / rank)
        q[i] = val
        prev = val
    out = np.empty(n, dtype=float)
    out[order] = q
    return out.tolist()

def format_p(p: float) -> str:
    if p is None or np.isnan(p):
        return 'n/a'
    if p < 1e-4:
        return '<1e-4'
    return f"{p:.4f}"

print('✓ Stats utilities ready')

✓ Stats utilities ready


In [38]:
# ============================================================================
# PRIMARY ASSOCIATION TEST BATTERY (robust to sparsity / missing columns)
# ============================================================================

tests: list[dict] = []

# Local fallbacks in case utilities were named differently upstream
if 'bh_fdr' not in globals():
    def bh_fdr(p_values: list[float]) -> list[float]:
        p = np.asarray(p_values, dtype=float)
        n = p.size
        order = np.argsort(p)
        ranked = p[order]
        q = ranked * n / (np.arange(n) + 1)
        q = np.minimum.accumulate(q[::-1])[::-1]
        q = np.clip(q, 0.0, 1.0)
        out = np.empty_like(q)
        out[order] = q
        return out.tolist()

if 'fmt_p' not in globals():
    def fmt_p(p: float) -> str:
        if p < 1e-4:
            return '< 1e-4'
        if p < 0.001:
            return f'{p:.4f}'
        return f'{p:.3f}'

def add_test(df: pd.DataFrame, x: str, y: str, **kwargs) -> None:
    title = kwargs.get('title', f'{x} × {y}')
    missing = [c for c in (x, y) if c not in df.columns]
    if missing:
        tests.append({
            'title': title,
            'x': x,
            'y': y,
            'n_total': None,
            'chi2': None,
            'dof': None,
            'p': None,
            'p_perm': None,
            'cramers_v': None,
            'v_ci_low': None,
            'v_ci_high': None,
            'notes': f"SKIPPED: missing columns {missing}",
        })
        return
    tests.append(association_report(df, x=x, y=y, **kwargs))

# Location-level associations
add_test(
    df_locations,
    x='location_type',
    y='atmosphere',
    min_total=200,
    drop_not_mentioned=True,
    max_levels=25,
    n_perm=500,
    n_boot=300,
    title='Location type × Atmosphere (top 25 levels)',
)

add_test(
    df_locations,
    x='vertical_position',
    y='atmosphere',
    min_total=200,
    drop_not_mentioned=True,
    max_levels=None,
    n_perm=500,
    n_boot=300,
    title='Vertical position × Atmosphere',
)

add_test(
    df_locations,
    x='light_temperature',
    y='atmosphere',
    min_total=120,
    drop_not_mentioned=True,
    max_levels=None,
    n_perm=500,
    n_boot=300,
    title='Light temperature × Atmosphere',
)

add_test(
    df_locations,
    x='reality_stability',
    y='atmosphere',
    min_total=200,
    drop_not_mentioned=True,
    max_levels=None,
    n_perm=500,
    n_boot=300,
    title='Reality stability × Atmosphere',
)

# Optional: only runs if the flattening produced this column
add_test(
    df_locations,
    x='water_presence',
    y='atmosphere',
    min_total=120,
    drop_not_mentioned=True,
    max_levels=12,
    n_perm=500,
    n_boot=300,
    title='Water presence × Atmosphere (top 12 levels)',
)

# Entity-level functional differentiation tests
add_test(
    df_entities,
    x='entity_type',
    y='demeanor',
    min_total=200,
    drop_not_mentioned=True,
    max_levels=20,
    n_perm=500,
    n_boot=300,
    title='Entity type × Demeanor',
)

add_test(
    df_entities,
    x='entity_type',
    y='authority_nature',
    min_total=200,
    drop_not_mentioned=True,
    max_levels=20,
    n_perm=500,
    n_boot=300,
    title='Entity type × Authority nature',
)

# Targeted subset: bathrooms (privacy × affect)
if 'location_type' in df_locations.columns:
    bathrooms = df_locations[df_locations['location_type'].eq('bathroom')].copy()
else:
    bathrooms = df_locations.iloc[0:0].copy()

add_test(
    bathrooms,
    x='privacy_status',
    y='affective_response',
    min_total=60,
    drop_not_mentioned=True,
    max_levels=None,
    n_perm=500,
    n_boot=300,
    title='Bathrooms only: Privacy status × Affective response',
)

df_tests = pd.DataFrame(tests)

# Multiple comparisons control on successfully computed p-values
mask_ok = df_tests['p'].notna()
if mask_ok.any():
    df_tests.loc[mask_ok, 'q_fdr_bh'] = bh_fdr(df_tests.loc[mask_ok, 'p'].to_list())
else:
    df_tests['q_fdr_bh'] = np.nan

display(
    df_tests.sort_values(['q_fdr_bh', 'p', 'cramers_v'], ascending=[True, True, False])
    .reset_index(drop=True)
    .assign(
        p=lambda d: d['p'].apply(lambda v: fmt_p(v) if pd.notna(v) else ''),
        p_perm=lambda d: d['p_perm'].apply(lambda v: fmt_p(v) if pd.notna(v) else ''),
        q_fdr_bh=lambda d: d['q_fdr_bh'].apply(lambda v: fmt_p(v) if pd.notna(v) else ''),
        cramers_v=lambda d: d['cramers_v'].round(3),
        v_ci_low=lambda d: d['v_ci_low'].round(3),
        v_ci_high=lambda d: d['v_ci_high'].round(3),
    )
)

,test,x,y,n,levels_x,levels_y,chi2,dof,p,cramers_v,p_perm,v_ci_low,v_ci_high,title,n_total,notes,q_fdr_bh
0,Entity type × Demeanor,entity_type,demeanor,2877.0,17.0,9.0,2125.604783,128.0,< 1e-4,0.295,0.002,0.281,0.321,NaN,NaN,NaN,< 1e-4
1,Location type × Atmosphere (top 25 levels),location_type,atmosphere,3521.0,25.0,10.0,945.423264,216.0,< 1e-4,0.152,0.002,0.157,0.186,NaN,NaN,NaN,< 1e-4
2,Reality stability × Atmosphere,reality_stability,atmosphere,2774.0,5.0,10.0,272.279737,36.0,< 1e-4,0.146,0.002,0.137,0.179,NaN,NaN,NaN,< 1e-4
3,Entity type × Authority nature,entity_type,authority_nature,651.0,14.0,5.0,277.166396,52.0,< 1e-4,0.295,0.002,0.279,0.346,NaN,NaN,NaN,< 1e-4
4,Light temperature × Atmosphere,light_temperature,atmosphere,604.0,3.0,10.0,184.457760,18.0,< 1e-4,0.372,0.002,0.330,0.433,NaN,NaN,NaN,< 1e-4
5,Vertical position × Atmosphere,vertical_position,atmosphere,1955.0,6.0,10.0,203.503150,45.0,< 1e-4,0.127,0.002,0.123,0.166,NaN,NaN,NaN,< 1e-4
6,NaN,water_presence,atmosphere,NaN,NaN,NaN,NaN,NaN,,NaN,,NaN,NaN,Water presence × Atmosphere (top 12 levels),NaN,SKIPPED: missing columns ['water_presence'],
7,Bathrooms only: Privacy status × Affective res...,privacy_status,affective_response,11.0,3.0,6.0,NaN,NaN,,NaN,,NaN,NaN,NaN,NaN,NaN,


In [40]:
# ============================================================================
# STRATIFIED STABILITY TESTS ("cultural" proxy splits + sparse demographics)
# ============================================================================

def stratified_v(
    df: pd.DataFrame,
    x: str,
    y: str,
    strata: pd.Series,
    *,
    min_n_per_stratum: int = 150,
    drop_not_mentioned: bool = True,
    max_levels: int | None = 25,
    n_perm: int = 2000,
    seed: int = 7,
) -> pd.DataFrame:
    """Compute Cramér's V per stratum and permutation-test the between-strata difference."""
    base = df[[x, y]].copy()
    base['__stratum__'] = strata.values
    if drop_not_mentioned:
        base = base[(base[x] != 'not_mentioned') & (base[y] != 'not_mentioned')]
    base = base.dropna()

    # Optional top-k collapsing done globally for comparability
    if max_levels is not None:
        top_x = base[x].value_counts().head(max_levels).index
        top_y = base[y].value_counts().head(max_levels).index
        base = base[base[x].isin(top_x) & base[y].isin(top_y)]

    # Compute per-stratum V
    rows = []
    for s, sub in base.groupby('__stratum__'):
        ct = pd.crosstab(sub[x], sub[y])
        n = int(ct.to_numpy().sum())
        if n < min_n_per_stratum or ct.shape[0] < 2 or ct.shape[1] < 2:
            continue
        v = cramers_v_bias_corrected(ct)
        rows.append({'stratum': s, 'n': n, 'cramers_v': v})
    out = pd.DataFrame(rows).sort_values('n', ascending=False)
    if len(out) < 2:
        return out

    # Permutation test: shuffle stratum labels, recompute max-min V spread
    rng = np.random.default_rng(seed)
    base = base.copy()
    observed_spread = float(out['cramers_v'].max() - out['cramers_v'].min())
    spreads = []
    stratum_labels = base['__stratum__'].to_numpy(dtype=object)
    for _ in range(n_perm):
        rng.shuffle(stratum_labels)
        base['__stratum__'] = stratum_labels
        vs = []
        for s, sub in base.groupby('__stratum__'):
            ct = pd.crosstab(sub[x], sub[y])
            n = int(ct.to_numpy().sum())
            if n < min_n_per_stratum or ct.shape[0] < 2 or ct.shape[1] < 2:
                continue
            vs.append(cramers_v_bias_corrected(ct))
        if len(vs) >= 2:
            spreads.append(float(np.max(vs) - np.min(vs)))
    if spreads:
        p_spread = float((np.sum(np.array(spreads) >= observed_spread) + 1) / (len(spreads) + 1))
        out['v_spread_observed'] = observed_spread
        out['v_spread_p_perm'] = p_spread
    return out

# Join dream-level strata to location table
dream_cols = ['dream_id', 'is_recent', 'has_image']
for c in ['author_gender', 'author_age_range', 'author_age', 'author_gender_inferred']:
    if c in df_dreams.columns:
        dream_cols.append(c)
loc_with_dream = df_locations.merge(df_dreams[dream_cols], on='dream_id', how='left')

def pick_col(df: pd.DataFrame, preferred: list[str]) -> str | None:
    for c in preferred:
        if c in df.columns:
            return c
    return None

gender_col = pick_col(loc_with_dream, ['author_gender', 'author_gender_y', 'author_gender_x', 'author_gender_inferred'])
age_col = pick_col(loc_with_dream, ['author_age_range', 'author_age_range_y', 'author_age_range_x', 'author_age'])

print('--- Location type × Atmosphere stability across time (recent vs earlier) ---')
display(stratified_v(
    loc_with_dream,
    x='location_type',
    y='atmosphere',
    strata=loc_with_dream['is_recent'].fillna(False),
    min_n_per_stratum=300,
    max_levels=25,
    n_perm=2000,
))

print('--- Vertical position × Atmosphere stability across time ---')
display(stratified_v(
    loc_with_dream,
    x='vertical_position',
    y='atmosphere',
    strata=loc_with_dream['is_recent'].fillna(False),
    min_n_per_stratum=300,
    max_levels=None,
    n_perm=2000,
))

if gender_col is not None:
    known_gender = loc_with_dream[loc_with_dream[gender_col] != 'not_mentioned'].copy()
    if len(known_gender) >= 300:
        print(f'--- Location type × Atmosphere by (sparse) gender column: {gender_col} ---')
        display(stratified_v(
            known_gender,
            x='location_type',
            y='atmosphere',
            strata=known_gender[gender_col],
            min_n_per_stratum=120,
            max_levels=25,
            n_perm=2000,
        ))
    else:
        print('Gender is too sparse for a reliable stratified test in this dataset run.')
else:
    print('No gender column available for stratified test.')

--- Location type × Atmosphere stability across time (recent vs earlier) ---


,stratum,n,cramers_v,v_spread_observed,v_spread_p_perm
0,False,2086,0.169041,0.008825,0.542229
1,True,1906,0.177867,0.008825,0.542229


--- Vertical position × Atmosphere stability across time ---


,stratum,n,cramers_v,v_spread_observed,v_spread_p_perm
0,False,1149,0.130410,0.013672,0.556722
1,True,1016,0.144082,0.013672,0.556722


Gender is too sparse for a reliable stratified test in this dataset run.


In [41]:
# ============================================================================
# COMPACT SUMMARY: strongest effects + what had enough data
# ============================================================================
cols = ['title','n_total','cramers_v','v_ci_low','v_ci_high','p','p_perm','q_fdr_bh','notes']
summary = df_tests.copy()
for c in cols:
    if c not in summary.columns:
        summary[c] = np.nan

summary['p_fmt'] = summary['p'].apply(lambda v: fmt_p(v) if pd.notna(v) else '')
summary['p_perm_fmt'] = summary['p_perm'].apply(lambda v: fmt_p(v) if pd.notna(v) else '')
summary['q_fmt'] = summary['q_fdr_bh'].apply(lambda v: fmt_p(v) if pd.notna(v) else '')

display(
    summary[cols + ['p_fmt','p_perm_fmt','q_fmt']]
    .sort_values(['q_fdr_bh','p','cramers_v'], ascending=[True, True, False])
    .head(15)
    .reset_index(drop=True)
)

print('--- Effect size guide (rough): V≈0.1 small, 0.3 medium, 0.5 large (context dependent) ---')
print('N tests computed:', int(summary['p'].notna().sum()), '/', len(summary))
print('Skipped tests:', int(summary['p'].isna().sum()))

,title,n_total,cramers_v,v_ci_low,v_ci_high,p,p_perm,q_fdr_bh,notes,p_fmt,p_perm_fmt,q_fmt
0,NaN,NaN,0.295012,0.281377,0.321187,0.000000e+00,0.001996,0.000000e+00,NaN,< 1e-4,0.002,< 1e-4
1,NaN,NaN,0.151905,0.157377,0.186345,8.094455e-92,0.001996,2.428337e-91,NaN,< 1e-4,0.002,< 1e-4
2,NaN,NaN,0.146026,0.136570,0.179158,4.563588e-38,0.001996,9.127176e-38,NaN,< 1e-4,0.002,< 1e-4
3,NaN,NaN,0.294913,0.278682,0.345741,1.785558e-32,0.001996,2.678336e-32,NaN,< 1e-4,0.002,< 1e-4
4,NaN,NaN,0.371793,0.329944,0.432997,1.252708e-29,0.001996,1.503250e-29,NaN,< 1e-4,0.002,< 1e-4
5,NaN,NaN,0.127493,0.123018,0.165857,4.972590e-22,0.001996,4.972590e-22,NaN,< 1e-4,0.002,< 1e-4
6,Water presence × Atmosphere (top 12 levels),NaN,NaN,NaN,NaN,NaN,NaN,NaN,SKIPPED: missing columns ['water_presence'],,,
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,


--- Effect size guide (rough): V≈0.1 small, 0.3 medium, 0.5 large (context dependent) ---
N tests computed: 6 / 8
Skipped tests: 2


In [44]:
# ============================================================================
# AUTHOR-BASED GENERALIZATION TESTS (shared structure vs storyteller effects)
# ============================================================================

def pick_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _prep_xy(
    df: pd.DataFrame,
    x: str,
    y: str,
    *,
    keep_cols: list[str] | None = None,
    drop_not_mentioned: bool = True,
    max_levels_x: int | None = None,
    max_levels_y: int | None = None,
) -> pd.DataFrame:
    cols = []
    if keep_cols:
        cols.extend([c for c in keep_cols if c in df.columns])
    cols.extend([x, y])
    cols = list(dict.fromkeys(cols))

    dfx = df[cols].copy()
    if drop_not_mentioned:
        dfx = dfx[(dfx[x] != 'not_mentioned') & (dfx[y] != 'not_mentioned')]
    dfx = dfx.dropna(subset=[x, y])
    if max_levels_x is not None:
        top_x = dfx[x].value_counts().head(max_levels_x).index
        dfx = dfx[dfx[x].isin(top_x)]
    if max_levels_y is not None:
        top_y = dfx[y].value_counts().head(max_levels_y).index
        dfx = dfx[dfx[y].isin(top_y)]
    return dfx


def compute_v_from_pairs(dfx: pd.DataFrame, x: str, y: str) -> float:
    ct = pd.crosstab(dfx[x], dfx[y])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return float('nan')
    return float(cramers_v_bias_corrected(ct))


def author_split_stability(
    df: pd.DataFrame,
    author_col: str,
    x: str,
    y: str,
    *,
    drop_not_mentioned: bool = True,
    max_levels_x: int | None = None,
    max_levels_y: int | None = None,
    n_splits: int = 200,
    test_frac: float = 0.5,
    seed: int = 7,
) -> pd.DataFrame:
    """Randomly split AUTHORS; compute V on each side. Repeats show stability."""
    if author_col not in df.columns:
        raise KeyError(f"author column not found: {author_col}")
    d = _prep_xy(
        df,
        x,
        y,
        keep_cols=[author_col],
        drop_not_mentioned=drop_not_mentioned,
        max_levels_x=max_levels_x,
        max_levels_y=max_levels_y,
    )
    d = d.dropna(subset=[author_col])

    authors = d[author_col].unique().tolist()
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n_splits):
        rng.shuffle(authors)
        cut = int(len(authors) * (1 - test_frac))
        train_auth = set(authors[:cut])
        test_auth = set(authors[cut:])
        d_train = d[d[author_col].isin(train_auth)]
        d_test = d[d[author_col].isin(test_auth)]
        v_train = compute_v_from_pairs(d_train, x, y)
        v_test = compute_v_from_pairs(d_test, x, y)
        rows.append({
            'n_authors': len(authors),
            'n_train': len(d_train),
            'n_test': len(d_test),
            'v_train': v_train,
            'v_test': v_test,
            'abs_diff': abs(v_train - v_test),
        })
    return pd.DataFrame(rows)


def drop_top_authors_sensitivity(
    df: pd.DataFrame,
    author_col: str,
    x: str,
    y: str,
    *,
    drop_not_mentioned: bool = True,
    max_levels_x: int | None = None,
    max_levels_y: int | None = None,
    drop_top_k_list: list[int] = [1, 5, 10, 25, 50],
) -> pd.DataFrame:
    """Recompute V after removing the most prolific authors."""
    if author_col not in df.columns:
        raise KeyError(f"author column not found: {author_col}")
    d = _prep_xy(
        df,
        x,
        y,
        keep_cols=[author_col],
        drop_not_mentioned=drop_not_mentioned,
        max_levels_x=max_levels_x,
        max_levels_y=max_levels_y,
    )
    d = d.dropna(subset=[author_col])

    counts = d[author_col].value_counts()
    rows = []
    base_v = compute_v_from_pairs(d, x, y)
    rows.append({'drop_top_k': 0, 'n': len(d), 'n_authors': int(d[author_col].nunique()), 'cramers_v': base_v})
    for k in drop_top_k_list:
        drop_auth = set(counts.head(k).index)
        sub = d[~d[author_col].isin(drop_auth)]
        rows.append({
            'drop_top_k': k,
            'n': len(sub),
            'n_authors': int(sub[author_col].nunique()),
            'cramers_v': compute_v_from_pairs(sub, x, y),
        })
    return pd.DataFrame(rows)


# Build location-level table with author (auto-detect author column name)
dream_author_col = pick_col(df_dreams, ['author', 'author_name', 'reddit_author', 'username', 'user', 'handle'])
if dream_author_col is None:
    print('No author column found in df_dreams; cannot run author-based tests.')
else:
    loc_author = df_locations.merge(df_dreams[['dream_id', dream_author_col]], on='dream_id', how='left')
    author_col_used = pick_col(loc_author, [dream_author_col, f'{dream_author_col}_x', f'{dream_author_col}_y', 'author'])

    if author_col_used is None:
        print('Author merge failed; cannot find merged author column.')
    else:
        print(f'Using author column: {author_col_used} (from df_dreams[{dream_author_col}])')

        print('AUTHOR-BASED STABILITY: location_type × atmosphere')
        stab1 = author_split_stability(
            loc_author,
            author_col=author_col_used,
            x='location_type',
            y='atmosphere',
            max_levels_x=25,
            max_levels_y=25,
            n_splits=300,
            test_frac=0.5,
)
        display(stab1.describe(percentiles=[0.1,0.5,0.9]).loc[['mean','std','min','10%','50%','90%','max']])

        print('Sensitivity to prolific authors (drop-top-k): location_type × atmosphere')
        sens1 = drop_top_authors_sensitivity(
            loc_author,
            author_col=author_col_used,
            x='location_type',
            y='atmosphere',
            max_levels_x=25,
            max_levels_y=25,
            drop_top_k_list=[1,5,10,25,50,100],
)
        display(sens1)

        print('AUTHOR-BASED STABILITY: vertical_position × atmosphere')
        stab2 = author_split_stability(
            loc_author,
            author_col=author_col_used,
            x='vertical_position',
            y='atmosphere',
            max_levels_x=None,
            max_levels_y=25,
            n_splits=300,
            test_frac=0.5,
)
        display(stab2.describe(percentiles=[0.1,0.5,0.9]).loc[['mean','std','min','10%','50%','90%','max']])

        print('Sensitivity to prolific authors (drop-top-k): vertical_position × atmosphere')
        sens2 = drop_top_authors_sensitivity(
            loc_author,
            author_col=author_col_used,
            x='vertical_position',
            y='atmosphere',
            max_levels_x=None,
            max_levels_y=25,
            drop_top_k_list=[1,5,10,25,50,100],
)
        display(sens2)

Using author column: author (from df_dreams[author])
AUTHOR-BASED STABILITY: location_type × atmosphere


,n_authors,n_train,n_test,v_train,v_test,abs_diff
mean,930.0,2002.593333,1989.406667,0.172417,0.171422,0.014360
std,0.0,81.697509,81.697509,0.009497,0.009339,0.011127
min,930.0,1825.000000,1725.000000,0.144969,0.140526,0.000022
10%,930.0,1895.800000,1887.000000,0.160031,0.160026,0.001563
50%,930.0,2008.500000,1983.500000,0.172691,0.171697,0.012646
90%,930.0,2105.000000,2096.200000,0.183930,0.182827,0.028958
max,930.0,2267.000000,2167.000000,0.202717,0.199165,0.062191


Sensitivity to prolific authors (drop-top-k): location_type × atmosphere


,drop_top_k,n,n_authors,cramers_v
0,0,3992,930,0.170411
1,1,3924,929,0.170669
2,5,3716,925,0.170897
3,10,3550,920,0.163860
4,25,3234,905,0.159019
5,50,2859,880,0.151658
6,100,2349,830,0.142356


AUTHOR-BASED STABILITY: vertical_position × atmosphere


,n_authors,n_train,n_test,v_train,v_test,abs_diff
mean,609.0,1080.963333,1084.036667,0.138779,0.136382,0.020989
std,0.0,62.074521,62.074521,0.014285,0.013747,0.016487
min,609.0,912.000000,921.000000,0.096765,0.095552,0.000049
10%,609.0,1003.500000,1007.900000,0.121318,0.120442,0.002526
50%,609.0,1088.500000,1076.500000,0.138951,0.136300,0.017465
90%,609.0,1157.100000,1161.500000,0.156087,0.152866,0.045857
max,609.0,1244.000000,1253.000000,0.173220,0.179121,0.076861


Sensitivity to prolific authors (drop-top-k): vertical_position × atmosphere


,drop_top_k,n,n_authors,cramers_v
0,0,2165,609,0.136036
1,1,2108,608,0.132648
2,5,1953,604,0.129767
3,10,1836,599,0.130658
4,25,1633,584,0.125076
5,50,1399,559,0.127932
6,100,1068,509,0.121387


In [45]:
# ============================================================================
# AUTHOR TESTS: compact numeric readout
# ============================================================================
def summarize_splits(df: pd.DataFrame, label: str) -> None:
    print(label)
    print('  splits:', len(df))
    print('  v_train mean ± sd:', float(df['v_train'].mean()), '±', float(df['v_train'].std()))
    print('  v_test  mean ± sd:', float(df['v_test'].mean()), '±', float(df['v_test'].std()))
    print('  abs_diff median:', float(df['abs_diff'].median()))
    print('  abs_diff 90%:', float(df['abs_diff'].quantile(0.9)))

if 'stab1' in globals():
    summarize_splits(stab1, 'location_type × atmosphere (author split)')
if 'stab2' in globals():
    summarize_splits(stab2, 'vertical_position × atmosphere (author split)')

if 'sens1' in globals():
    print('\nDrop-top-k (location_type × atmosphere):')
    display(sens1)
if 'sens2' in globals():
    print('\nDrop-top-k (vertical_position × atmosphere):')
    display(sens2)

location_type × atmosphere (author split)
  splits: 300
  v_train mean ± sd: 0.1724167577368361 ± 0.009496932591190624
  v_test  mean ± sd: 0.17142243836034396 ± 0.009338580907512774
  abs_diff median: 0.012645914726336277
  abs_diff 90%: 0.028958381071825266
vertical_position × atmosphere (author split)
  splits: 300
  v_train mean ± sd: 0.1387785484266604 ± 0.01428497267013598
  v_test  mean ± sd: 0.13638205981745918 ± 0.013746654028462677
  abs_diff median: 0.01746546019129109
  abs_diff 90%: 0.045856704654233034

Drop-top-k (location_type × atmosphere):


,drop_top_k,n,n_authors,cramers_v
0,0,3992,930,0.170411
1,1,3924,929,0.170669
2,5,3716,925,0.170897
3,10,3550,920,0.163860
4,25,3234,905,0.159019
5,50,2859,880,0.151658
6,100,2349,830,0.142356



Drop-top-k (vertical_position × atmosphere):


,drop_top_k,n,n_authors,cramers_v
0,0,2165,609,0.136036
1,1,2108,608,0.132648
2,5,1953,604,0.129767
3,10,1836,599,0.130658
4,25,1633,584,0.125076
5,50,1399,559,0.127932
6,100,1068,509,0.121387


In [48]:
# ============================================================================
# WITHIN-AUTHOR PERMUTATION NULL (controls for author style/base rates)
# ============================================================================

def within_author_permutation_v(
    df: pd.DataFrame,
    author_col: str,
    x: str,
    y: str,
    *,
    drop_not_mentioned: bool = True,
    max_levels_x: int | None = None,
    max_levels_y: int | None = None,
    n_perm: int = 1000,
    seed: int = 7,
) -> dict:
    if author_col not in df.columns:
        raise KeyError(f"author column not found: {author_col}")

    d = df[[author_col, x, y]].copy()
    if drop_not_mentioned:
        d = d[(d[x] != 'not_mentioned') & (d[y] != 'not_mentioned')]
    d = d.dropna(subset=[author_col, x, y])

    # Global top-k filtering for stable label set
    if max_levels_x is not None:
        top_x = d[x].value_counts().head(max_levels_x).index
        d = d[d[x].isin(top_x)]
    if max_levels_y is not None:
        top_y = d[y].value_counts().head(max_levels_y).index
        d = d[d[y].isin(top_y)]

    # Important: reindex to 0..n-1 so positional indexing works safely
    d = d.reset_index(drop=True)

    observed_v = compute_v_from_pairs(d, x, y)
    rng = np.random.default_rng(seed)

    # Pre-split positional indices per author for speed
    groups = [sub.index.to_numpy() for _, sub in d.groupby(author_col)]
    y_values = d[y].to_numpy(dtype=object)

    null_vs = []
    for _ in range(n_perm):
        y_perm = y_values.copy()
        for idx in groups:
            if idx.size > 1:
                # NOTE: y_perm[idx] is a copy; must assign permuted values back
                y_perm[idx] = rng.permutation(y_perm[idx])
        d_perm = d.copy()
        d_perm[y] = y_perm
        null_vs.append(compute_v_from_pairs(d_perm, x, y))

    null_vs_arr = np.asarray(null_vs, dtype=float)
    p = float((np.sum(null_vs_arr >= observed_v) + 1) / (len(null_vs_arr) + 1))

    return {
        'x': x,
        'y': y,
        'n': int(len(d)),
        'n_authors': int(d[author_col].nunique()),
        'observed_v': float(observed_v),
        'null_mean_v': float(np.nanmean(null_vs_arr)),
        'null_p90_v': float(np.nanquantile(null_vs_arr, 0.9)),
        'null_max_v': float(np.nanmax(null_vs_arr)),
        'p_perm_within_author': p,
    }

if 'loc_author' in globals() and 'author_col_used' in globals():
    print('WITHIN-AUTHOR NULL: location_type × atmosphere')
    r1 = within_author_permutation_v(
        loc_author,
        author_col=author_col_used,
        x='location_type',
        y='atmosphere',
        max_levels_x=25,
        max_levels_y=25,
        n_perm=1000,
        seed=7,
    )
    display(pd.DataFrame([r1]))

    print('WITHIN-AUTHOR NULL: vertical_position × atmosphere')
    r2 = within_author_permutation_v(
        loc_author,
        author_col=author_col_used,
        x='vertical_position',
        y='atmosphere',
        max_levels_x=None,
        max_levels_y=25,
        n_perm=1000,
        seed=7,
    )
    display(pd.DataFrame([r2]))
else:
    print('loc_author/author_col_used not available; run the author-based tests cell first.')

WITHIN-AUTHOR NULL: location_type × atmosphere


,x,y,n,n_authors,observed_v,null_mean_v,null_p90_v,null_max_v,p_perm_within_author
0,location_type,atmosphere,3992,930,0.170411,0.059292,0.0678,0.080633,0.000999


WITHIN-AUTHOR NULL: vertical_position × atmosphere


,x,y,n,n_authors,observed_v,null_mean_v,null_p90_v,null_max_v,p_perm_within_author
0,vertical_position,atmosphere,2165,609,0.136036,0.059965,0.073014,0.09335,0.000999


In [49]:
from scipy.stats import chi2_contingency

def _chi2_summary(df: pd.DataFrame, x: str, y: str) -> dict:
    d = df[[x, y]].dropna().copy()
    d[x] = d[x].astype(str)
    d[y] = d[y].astype(str)
    d = d[(d[x] != "not_mentioned") & (d[y] != "not_mentioned")]
    ct = pd.crosstab(d[x], d[y])
    chi2, p, dof, _ = chi2_contingency(ct)
    n = int(ct.values.sum())
    r, c = ct.shape
    k = min(r, c)
    v = float("nan") if (n == 0 or k <= 1) else float((chi2 / (n * (k - 1))) ** 0.5)
    return {
        "x": x,
        "y": y,
        "n": n,
        "r": r,
        "c": c,
        "chi2": float(chi2),
        "dof": int(dof),
        "p": float(p),
        "cramers_v": v,
    }

print("Primary chi-square summaries (as used in-notebook):")
for x, y in [("location_type", "atmosphere"), ("vertical_position", "atmosphere")]:
    s = _chi2_summary(df_locations, x, y)
    print(
        f"- {s['x']} × {s['y']}: n={s['n']}, r×c={s['r']}×{s['c']}, "
        f"χ²={s['chi2']:.2f}, df={s['dof']}, p={s['p']:.3e}, V={s['cramers_v']:.3f}"
    )


Primary chi-square summaries (as used in-notebook):
- location_type × atmosphere: n=4306, r×c=70×10, χ²=1737.99, df=621, p=2.126e-106, V=0.212
- vertical_position × atmosphere: n=1955, r×c=6×10, χ²=203.50, df=45, p=4.973e-22, V=0.144
